### Cấu hình thư viện và môi trường

In [12]:

try:
    import threadpoolctl
    orig_init = threadpoolctl.LibController.__init__
    def safe_init(self, *, filepath=None, prefix=None, parent=None):
        try:
            orig_init(self, filepath=filepath, prefix=prefix, parent=parent)
        except Exception:
            self.parent, self.prefix, self.filepath = parent, prefix, filepath
            self.dynlib = None
            self._symbol_prefix, self._symbol_suffix = '', ''
            self.version = None
    threadpoolctl.LibController.__init__ = safe_init
except Exception:
    pass

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Đã nạp toàn bộ thư viện thành công!')

Đã nạp toàn bộ thư viện thành công!


In [13]:
# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    'A': np.array([5.0, 3.0, 4000.0]),
    'B': np.array([25.0, 9.0, 4100.0]),
    'C': np.array([6.0, 3.5, 7000.0]),
}

def kc(u, v):
    return np.sqrt(((u - v) ** 2).sum())

print('Khoang cach KHI CHUA chuan hoa')
print(f'  d(A, B) = {kc(ba_ban["A"], ba_ban["B"]):8.1f}')
print(f'  d(A, C) = {kc(ba_ban["A"], ba_ban["C"]):8.1f}')
print()

df_sv = pd.read_csv('data/sinh_vien_mo_rong.csv')
X = df_sv[['gio_on', 'diem_giua_ky', 'chi_tieu']].to_numpy(dtype=float)
tb, dl = X.mean(axis=0), X.std(axis=0)

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}
print('Khoang cach SAU KHI chuan hoa')
print(f'  d(A, B) = {kc(chuan["A"], chuan["B"]):.2f}')
print(f'  d(A, C) = {kc(chuan["A"], chuan["C"]):.2f}')
print()

# Tìm 5 láng giềng cho bạn mới
X2 = df_sv[['gio_on', 'diem_giua_ky']].to_numpy(dtype=float)
y_sv = df_sv['qua_mon'].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])
X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2
khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print('Ban moi: on 12 gio, diem giua ky 6.5')
print('Nam lang gieng gan nhat:')
for i in gan:
    ket = 'qua' if y_sv[i] == 1 else 'rot'
    print(f'  gio {X2[i, 0]:5.1f}  diem {X2[i, 1]:4.1f}  khoang cach {khoang[i]:.3f} -> {ket}')
so_qua = int(y_sv[gan].sum())
print(f'Bo phieu: {so_qua} qua, {5 - so_qua} rot')
print('Du doan:', 'qua mon' if so_qua >= 3 else 'rot mon')

Khoang cach KHI CHUA chuan hoa
  d(A, B) =    102.2
  d(A, C) =   3000.0

Khoang cach SAU KHI chuan hoa
  d(A, B) = 3.85
  d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
  gio  10.5  diem  6.4  khoang cach 0.198 -> qua
  gio  12.5  diem  6.1  khoang cach 0.202 -> qua
  gio  14.0  diem  6.4  khoang cach 0.260 -> qua
  gio   9.5  diem  6.1  khoang cach 0.373 -> rot
  gio  10.5  diem  5.8  khoang cach 0.386 -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


--- 
## 2:ĐO HẬU QUẢ KHI QUÊN CHUẨN HOÁ 

In [14]:
y = df_sv['qua_mon']
X3 = df_sv[['gio_on', 'diem_giua_ky', 'chi_tieu']]
X_tr_sv, X_te_sv, y_tr_sv, y_te_sv = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)

def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr_sv[cot], y_tr_sv)
    return accuracy_score(y_te_sv, mo_hinh.predict(X_te_sv[cot]))

def knn_chuan():
    return make_pipeline(StandardScaler(), KNeighborsClassifier(7))

ba_cot = ['gio_on', 'diem_giua_ky', 'chi_tieu']
hai_cot = ['gio_on', 'diem_giua_ky']

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

print(f'Doan bua ca lop qua          : {y_te_sv.mean():.4f}')
print(f'Ba cot, khong chuan hoa      : {a:.4f}')
print(f'Ba cot, co chuan hoa         : {b:.4f}')
print(f'Bo cot chi_tieu, co chuan hoa: {c:.4f}')

Doan bua ca lop qua          : 0.6000
Ba cot, khong chuan hoa      : 0.5333
Ba cot, co chuan hoa         : 0.8667
Bo cot chi_tieu, co chuan hoa: 0.9000


--- 
## 3: CHỌN SỐ LÁNG GIỀNG K BẰNG KIỂM ĐỊNH CHÉO 

- Bộ dữ liệu hai vầng trăng khuyết lồng vào nhau.
- Đánh giá ảnh hưởng của $k$ (học vẹt khi $k=1$, cẩu thả khi $k=101$).
- Dùng kiểm định chéo 5-fold trên tập học để tìm $k$ tối ưu.

In [15]:
df_moon = pd.read_csv('data/hai_vang_trang.csv')
X_m = df_moon[['x1', 'x2']]
y_m = df_moon['nhan']

X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_m, y_m, test_size=0.25, random_state=17, stratify=y_m
)

print('Accuracy tren tap hoc va tap kiem tra theo k:')
print('   k  tap hoc  tap kiem tra')
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr_m, y_tr_m)
    a_hoc = accuracy_score(y_tr_m, mh.predict(X_tr_m))
    a_kt = accuracy_score(y_te_m, mh.predict(X_te_m))
    print(f' {k:3d}   {a_hoc:.4f}        {a_kt:.4f}')
print()

print('Kiem dinh cheo 5 phan tren tap hoc:')
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_tr_m, y_tr_m, cv=5).mean()

k_tot = max(diem, key=diem.get)
for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = ' <- tot nhat' if k == k_tot else ''
    print(f'  k = {k:2d}  diem trung binh {diem[k]:.4f}{dau}')
print()

mh_tot = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr_m, y_tr_m)
acc_test = accuracy_score(y_te_m, mh_tot.predict(X_te_m))
print(f'k tot nhat = {k_tot}')
print(f'Accuracy tren tap kiem tra = {acc_test:.4f}')

Accuracy tren tap hoc va tap kiem tra theo k:
   k  tap hoc  tap kiem tra
   1   1.0000        0.8000
   5   0.8756        0.9067
  13   0.8844        0.9200
  31   0.8667        0.8933
 101   0.8311        0.8533

Kiem dinh cheo 5 phan tren tap hoc:
  k =  1  diem trung binh 0.7778
  k =  5  diem trung binh 0.8444
  k =  9  diem trung binh 0.8444
  k = 13  diem trung binh 0.8667 <- tot nhat
  k = 17  diem trung binh 0.8533
  k = 25  diem trung binh 0.8622
  k = 51  diem trung binh 0.8533

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


--- 
## 4: SO SÁNH KNN VỚI HỒI QUY LOGISTIC VÀ NAIVE BAYES 

In [16]:
cac_mo_hinh = [
    ('Hoi quy logistic', LogisticRegression()),
    ('Naive Bayes    ', GaussianNB()),
    ('KNN, k = 13     ', KNeighborsClassifier(n_neighbors=13)),
]

print('Accuracy tren tap kiem tra, bo hai vang trang')
for ten, mh in cac_mo_hinh:
    mh.fit(X_tr_m, y_tr_m)
    print(f'  {ten}: {accuracy_score(y_te_m, mh.predict(X_te_m)):.4f}')

Accuracy tren tap kiem tra, bo hai vang trang
  Hoi quy logistic: 0.8533
  Naive Bayes    : 0.8533
  KNN, k = 13     : 0.9200


--- 
## 5: KNN CHO BÀI TOÁN HỒI QUY & KHÔNG THỂ NGOẠI SUY 

In [17]:
df_nha = pd.read_csv('data/gia_nha.csv')
X_nha = df_nha[['dien_tich']]
y_nha = df_nha['gia']

X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(X_nha, y_nha, test_size=0.2, random_state=42)

knn_reg = KNeighborsRegressor(n_neighbors=5).fit(X_tr_n, y_tr_n)
duong = LinearRegression().fit(X_tr_n, y_tr_n)

print('MAE tren tap kiem tra (ty dong)')
print(f'  KNN, k = 5   : {mean_absolute_error(y_te_n, knn_reg.predict(X_te_n)):.4f}')
print(f'  Duong thang  : {mean_absolute_error(y_te_n, duong.predict(X_te_n)):.4f}')
print()

can_moi = pd.DataFrame({'dien_tich': [60.0, 100.0, 150.0, 250.0]})
print('Du doan cho bon can moi (ty dong)')
print('  dien tich    KNN  duong thang')
for dt, a, b in zip(can_moi['dien_tich'], knn_reg.predict(can_moi), duong.predict(can_moi)):
    print(f'  {dt:6.0f} m2   {a:6.2f}        {b:6.2f}')
print()

_, chi_so = knn_reg.kneighbors(pd.DataFrame({'dien_tich': [250.0]}))
lg = X_tr_n.iloc[chi_so[0]]['dien_tich'].tolist()
print('Nam lang gieng cua can 250 m2:', ', '.join(f'{v:.1f}' for v in lg))

MAE tren tap kiem tra (ty dong)
  KNN, k = 5   : 0.4030
  Duong thang  : 0.2578

Du doan cho bon can moi (ty dong)
  dien tich    KNN  duong thang
      60 m2     4.95          5.12
     100 m2     8.32          8.24
     150 m2     9.32         12.13
     250 m2     9.32         19.91

Nam lang gieng cua can 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2



## 6. BÀI TẬP 1: TỰ VIẾT KNN KHÔNG DÙNG THƯ VIỆN 



In [18]:
def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    # 1. Tính khoảng cách Euclid từ x_moi đến mọi điểm trong X_hoc
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))
    # 2. Lấy k chỉ số gần nhất
    chi_so_k = np.argsort(khoang_cach)[:k]
    # 3. Bỏ phiếu đa số
    nhan_k = y_hoc[chi_so_k]
    return np.bincount(nhan_k).argmax()

X_tr_arr, X_te_arr = X_tr_m.to_numpy(), X_te_m.to_numpy()
y_tr_arr, y_te_arr = y_tr_m.to_numpy(), y_te_m.to_numpy()

k = 13
y_pred_tu_viet = np.array([knn_du_doan(X_tr_arr, y_tr_arr, x, k) for x in X_te_arr])
acc_tu_viet = accuracy_score(y_te_arr, y_pred_tu_viet)

mo_hinh_sklearn = KNeighborsClassifier(n_neighbors=k).fit(X_tr_m, y_tr_m)
acc_sklearn = accuracy_score(y_te_m, mo_hinh_sklearn.predict(X_te_m))

print(f'Accuracy KNN tự viết (k = {k})         : {acc_tu_viet:.4f}')
print(f'Accuracy KNeighborsClassifier (k = {k}) : {acc_sklearn:.4f}')
print(f'Hai kết quả có bằng nhau tuyệt đối không? : {acc_tu_viet == acc_sklearn}')

Accuracy KNN tự viết (k = 13)         : 0.9200
Accuracy KNeighborsClassifier (k = 13) : 0.9200
Hai kết quả có bằng nhau tuyệt đối không? : True


--- 
## 7. BÀI TẬP 2: KHOẢNG CÁCH MANHATTAN 


In [19]:
diem_manhattan = {}
for k in range(1, 52, 2):
    diem_manhattan[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, metric='manhattan'),
        X_tr_m, y_tr_m, cv=5
    ).mean()

k_tot_manhattan = max(diem_manhattan, key=diem_manhattan.get)
mh_manhattan = KNeighborsClassifier(n_neighbors=k_tot_manhattan, metric='manhattan').fit(X_tr_m, y_tr_m)
acc_test_manhattan = accuracy_score(y_te_m, mh_manhattan.predict(X_te_m))

mh_euclid = KNeighborsClassifier(n_neighbors=13, metric='euclidean').fit(X_tr_m, y_tr_m)
acc_test_euclid = accuracy_score(y_te_m, mh_euclid.predict(X_te_m))

print(f'Manhattan: k tốt nhất = {k_tot_manhattan}, CV = {diem_manhattan[k_tot_manhattan]:.4f}, Test Accuracy = {acc_test_manhattan:.4f}')
print(f'Euclid   : k tốt nhất = 13, CV = 0.8667, Test Accuracy = {acc_test_euclid:.4f}')
print('\nNhận xét: Đổi cách đo từ Euclid sang Manhattan không làm kết quả khác đi nhiều (90.67% so với 92.00%) vì hình học hai vầng trăng bảo toàn thứ tự láng giềng.')

Manhattan: k tốt nhất = 27, CV = 0.8667, Test Accuracy = 0.9067
Euclid   : k tốt nhất = 13, CV = 0.8667, Test Accuracy = 0.9200

Nhận xét: Đổi cách đo từ Euclid sang Manhattan không làm kết quả khác đi nhiều (90.67% so với 92.00%) vì hình học hai vầng trăng bảo toàn thứ tự láng giềng.


--- 
## 8. BÀI TẬP 3: ĐỔI ĐƠN VỊ ĐO 

In [20]:
df_sv_moi = df_sv.copy()
df_sv_moi['chi_tieu_trieu'] = df_sv_moi['chi_tieu'] / 1000.0

X_cu = df_sv[['gio_on', 'diem_giua_ky', 'chi_tieu']]
X_moi = df_sv_moi[['gio_on', 'diem_giua_ky', 'chi_tieu_trieu']]
y = df_sv['qua_mon']

X_tr_c, X_te_c, y_tr, y_te = train_test_split(X_cu, y, test_size=0.25, random_state=17, stratify=y)
X_tr_moi, X_te_moi, _, _ = train_test_split(X_moi, y, test_size=0.25, random_state=17, stratify=y)

m_cu_kchuan = KNeighborsClassifier(n_neighbors=7).fit(X_tr_c, y_tr)
m_moi_kchuan = KNeighborsClassifier(n_neighbors=7).fit(X_tr_moi, y_tr)

pipe_cu = make_pipeline(StandardScaler(), KNeighborsClassifier(7)).fit(X_tr_c, y_tr)
pipe_moi = make_pipeline(StandardScaler(), KNeighborsClassifier(7)).fit(X_tr_moi, y_tr)

print(f'Nghìn đồng (Không chuẩn hóa): {accuracy_score(y_te, m_cu_kchuan.predict(X_te_c)):.4f}')
print(f'Triệu đồng (Không chuẩn hóa): {accuracy_score(y_te, m_moi_kchuan.predict(X_te_moi)):.4f}')
print(f'Nghìn đồng (CÓ chuẩn hóa)   : {accuracy_score(y_te, pipe_cu.predict(X_te_c)):.4f}')
print(f'Triệu đồng (CÓ chuẩn hóa)   : {accuracy_score(y_te, pipe_moi.predict(X_te_moi)):.4f}')

Nghìn đồng (Không chuẩn hóa): 0.5333
Triệu đồng (Không chuẩn hóa): 0.8667
Nghìn đồng (CÓ chuẩn hóa)   : 0.8667
Triệu đồng (CÓ chuẩn hóa)   : 0.8667


--- 
## 9. BÀI TẬP 4: LÁNG GIỀNG CÓ TRỌNG SỐ 


In [21]:
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    df_moon[['x1', 'x2']], df_moon['nhan'], test_size=0.25, random_state=17, stratify=df_moon['nhan']
)

print(f"{'k':>5} | {'Uniform Train':>15} | {'Uniform Test':>15} | {'Distance Train':>15} | {'Distance Test':>15}")
print('-' * 75)
for k in [1, 13, 101]:
    u = KNeighborsClassifier(n_neighbors=k, weights='uniform').fit(X_tr_m, y_tr_m)
    d = KNeighborsClassifier(n_neighbors=k, weights='distance').fit(X_tr_m, y_tr_m)
    print(f"{k:5d} | {accuracy_score(y_tr_m, u.predict(X_tr_m)):15.4f} | {accuracy_score(y_te_m, u.predict(X_te_m)):15.4f} | {accuracy_score(y_tr_m, d.predict(X_tr_m)):15.4f} | {accuracy_score(y_te_m, d.predict(X_te_m)):15.4f}")

    k |   Uniform Train |    Uniform Test |  Distance Train |   Distance Test
---------------------------------------------------------------------------
    1 |          1.0000 |          0.8000 |          1.0000 |          0.8000
   13 |          0.8844 |          0.9200 |          1.0000 |          0.9333
  101 |          0.8311 |          0.8533 |          1.0000 |          0.9067


--- 
## 10. BÀI TẬP 5: CHỌN K CHO BÀI TOÁN HỒI QUY GIÁ NHÀ 


In [22]:
diem_mae = {}
for k in range(1, 21):
    sc = cross_val_score(KNeighborsRegressor(n_neighbors=k), X_tr_n, y_tr_n, cv=5, scoring='neg_mean_absolute_error')
    diem_mae[k] = -sc.mean()

k_tot_reg = min(diem_mae, key=diem_mae.get)
print(f'Giá trị k tối ưu theo CV 5-fold: k = {k_tot_reg} (MAE CV = {diem_mae[k_tot_reg]:.4f} tỷ đồng)\n')

knn_tot = KNeighborsRegressor(n_neighbors=k_tot_reg).fit(X_tr_n, y_tr_n)
mae_knn = mean_absolute_error(y_te_n, knn_tot.predict(X_te_n))
mae_duong = mean_absolute_error(y_te_n, duong.predict(X_te_n))

print(f'MAE Test của KNN (k = {k_tot_reg}) : {mae_knn:.4f} tỷ đồng')
print(f'MAE Test của Đường thẳng   : {mae_duong:.4f} tỷ đồng')
print('-> Kết luận: KNN không thể vượt được Hồi quy tuyến tính vì giá nhà tăng tuyến tính hoàn hảo theo diện tích.')

Giá trị k tối ưu theo CV 5-fold: k = 2 (MAE CV = 0.3498 tỷ đồng)

MAE Test của KNN (k = 2) : 0.4867 tỷ đồng
MAE Test của Đường thẳng   : 0.2578 tỷ đồng
-> Kết luận: KNN không thể vượt được Hồi quy tuyến tính vì giá nhà tăng tuyến tính hoàn hảo theo diện tích.


c:\Users\nthie\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\nthie\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\nthie\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\nthie\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", l

--- 
## 11. BÀI TẬP 6: ĐO THỜI GIAN FIT VÀ PREDICT - BẢN CHẤT LAZY LEARNING 


In [23]:
try:
    from IPython.display import display
except ImportError:
    display = print

rng = np.random.default_rng(0)
X_moi = rng.random((1000, 2))

bang_kq = []
for n in [1000, 10000, 100000]:
    X_h = rng.random((n, 2))
    y_h = rng.integers(0, 2, size=n)
    
    t0 = time.perf_counter()
    clf = KNeighborsClassifier(n_neighbors=5).fit(X_h, y_h)
    t_fit = (time.perf_counter() - t0) * 1000
    
    t1 = time.perf_counter()
    clf.predict(X_moi)
    t_pred = (time.perf_counter() - t1) * 1000
    
    bang_kq.append({'Số mẫu (N)': f'{n:,}', 'Thời gian fit (ms)': f'{t_fit:.3f}', 'Thời gian predict 1000 mẫu (ms)': f'{t_pred:.3f}'})

df_time = pd.DataFrame(bang_kq)
display(df_time)

,Số mẫu (N),Thời gian fit (ms),Thời gian predict 1000 mẫu (ms)
0,"1,000",1.086,3.298
1,"10,000",4.767,3.585
2,"100,000",58.969,4.723
